# Optimization

## Import Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import config
import src.optimization_utils as ou
import src.solutions as s
import src.display as disp
import src.runs as runs
from src.problem import SpineProblem

# Reload to pick up any changes
importlib.reload(ou)
importlib.reload(s)
importlib.reload(disp)
importlib.reload(runs)

pd.set_option('display.max_columns', None)

In [2]:
# =============================================================================
# GA Configuration
# =============================================================================
POP_SIZE  = 100   # GA population size
N_GEN     = 10    # Number of generations
SEED      = 42    # Random seed for reproducibility
TOP_N     = 4     # Number of diverse solutions to display
SCORE_TOL = 5     # Score window around best for diversity selection

PSO_LL_OVERRIDE = False  # Toggle clinical delta_LL correction for literature ranges
DEDUP_BEST = True          # Deduplicate best plans across scenarios


## Load Models

In [3]:
# Load all models
model_configs = {
    "Mechanical Failure": config.MECH_FAIL_MODEL,
    "L4S1": config.L4S1_MODEL,
    "LL": config.LL_MODEL,
    "T4PA": config.T4PA_MODEL,
    "L1PA": config.L1PA_MODEL,
    "SVA": config.SVA_MODEL,
    "SS": config.SS_MODEL,
    "Global Tilt": config.GLOBAL_TILT_MODEL,
    "ODI": config.ODI_MODEL,
}

bundles = {name: ou.load_model_bundle(path) for name, path in model_configs.items()}

# Verify and display
print("Models loaded and verified:")
for name, bundle in bundles.items():
    status = "✓" if all(k in bundle for k in ["pipe", "features", "target"]) else "✗"
    print(f"  {status} {name}: {bundle.get('model_name', 'N/A')}")

# Extract individual bundles for use
mech_fail_bundle = bundles["Mechanical Failure"]
L4S1_bundle = bundles["L4S1"]
LL_bundle = bundles["LL"]
T4PA_bundle = bundles["T4PA"]
L1PA_bundle = bundles["L1PA"]
SVA_bundle = bundles["SVA"]
SS_bundle = bundles["SS"]
GT_bundle = bundles["Global Tilt"]
ODI_bundle = bundles["ODI"]

# Delta model bundles for postop predictions
delta_bundles = {
    "L4S1": L4S1_bundle,
    "LL": LL_bundle,
    "T4PA": T4PA_bundle,
    "L1PA": L1PA_bundle,
    "SS": SS_bundle,
    "GlobalTilt": GT_bundle,
    "SVA": SVA_bundle,
}

Models loaded and verified:
  ✓ Mechanical Failure: mech_fail_logreg
  ✓ L4S1: Ridge
  ✓ LL: RandomForest
  ✓ T4PA: Ridge
  ✓ L1PA: Ridge
  ✓ SVA: XGBRegressor
  ✓ SS: XGBRegressor
  ✓ Global Tilt: XGBRegressor
  ✓ ODI: Ridge


### Solution Space (Decision Variable Bounds)

In [4]:
UIV_CHOICES, xl, xu = ou.get_decision_config()

# Build a formatted solution space table
bounds_df = pd.DataFrame(
    {"Lower Bound": xl, "Upper Bound": xu},
    index=config.DECISION_VAR_NAMES,
)

# Add a "Type" column and decode categorical bounds
var_types = []
for spec in config.DECISION_VAR_SPECS:
    if spec.get("categorical"):
        var_types.append(f"Categorical: {spec['choices']}")
    elif spec["upper"] == 1:
        var_types.append("Binary (0/1)")
    else:
        var_types.append("Integer")
bounds_df.insert(0, "Type", var_types)

display(Markdown("**Decision Variable Bounds**"))
display(bounds_df)

**Decision Variable Bounds**

,Type,Lower Bound,Upper Bound
uiv_code,"Categorical: ['Hook', 'PS', 'FS']",0,2
num_levels_cat_code,"Categorical: ['lower', 'higher']",0,1
num_interbody_fusion_levels,Integer,0,5
ALIF,Binary (0/1),0,1
XLIF,Binary (0/1),0,1
TLIF,Binary (0/1),0,1
num_rods,Integer,1,4
num_pelvic_screws,Integer,2,4
osteotomy,Binary (0/1),0,1


## Optimization Scenarios

Each scenario uses a different set of composite score weights to explore how the optimizer behaves under different objectives. Predefined presets are in `src/runs.py`. To add a new scenario, add a new entry to `runs.PRESETS` and include its key below.

In [5]:
# =============================================================================
# SELECT SCENARIOS TO RUN
# =============================================================================
# Available presets (defined in src/runs.py):
#   "equal"           - All 6 alignment components weighted equally
#   "mech_fail"       - Primarily mech failure, small alignment weights
#   "mech_fail_t4l1pa"- Mech failure + T4PA-L1PA mismatch
#   "l4s1"            - Primarily L4-S1 in ideal range
#   "t4l1pa"          - Primarily T4PA-L1PA mismatch
#   "equal_plus_mech" - Composite (equal) blended 50/50 with mech failure
#   "odi"             - Primarily lowest predicted postop ODI
#   "gap_score"       - Primarily GAP score + category improvement
#   "gap_score_mech_fail" - GAP score + Mechanical Failure
#   "ll"              - Primarily PI-LL mismatch
#
# To add a new scenario: add an entry to runs.PRESETS in src/runs.py,
# then include its key here.

SCENARIOS = ["equal", "mech_fail", "mech_fail_t4l1pa", "l4s1", "t4l1pa", "gap_score", "gap_score_mech_fail", "ll", "odi"]

# Show selected scenarios
for key in SCENARIOS:
    runs.print_preset(runs.PRESETS[key])


  Equal Weights (Composite)
  All 6 alignment components and mechanical failure weighted equally
                       GAP Score: 0.1429 ◀
                    L1PA penalty: 0.1429 ◀
                    L4S1 penalty: 0.1429 ◀
                  T4L1PA penalty: 0.1429 ◀
                      LL penalty: 0.1429 ◀
        GAP category improvement: 0.1429 ◀
         Mechanical failure prob: 0.1429 ◀

  Minimize Mechanical Failure
  Primarily mechanical failure probability, with small alignment weights
                       GAP Score: 0.0500 ◀
                    L1PA penalty: 0.0500 ◀
                    L4S1 penalty: 0.0500 ◀
                  T4L1PA penalty: 0.0500 ◀
                      LL penalty: 0.0500 ◀
        GAP category improvement: 0.0500 ◀
         Mechanical failure prob: 0.7000 ◀

  Minimize Mechanical Failure + T4L1PA
  Mechanical failure probability and T4PA-L1PA mismatch weighted equally
                       GAP Score: 0.0000
                    L1PA penalty: 0.0000
 

## All Holdout Patients

In [6]:
# Show holdout patients summary
holdout_df = pd.read_csv(config.DATA_HOLDOUT)

summary_cols = [
    'sex', 'age', 'ASA_CLASS',
    "id", "revision", "mech_fail_last",
    "gap_score_preop", "gap_category", "gap_score_postop", "gap_category_postop",
    "UIV_implant", "num_levels_cat", "num_interbody_fusion_levels",
    "ALIF", "XLIF", "TLIF", "num_rods", "num_pelvic_screws", "osteotomy",
    "ODI_preop"
]
display(holdout_df[summary_cols].set_index("id"))

# =============================================================================
# RUN OPTIMIZATION FOR ALL HOLDOUT PATIENTS
# =============================================================================
all_patient_results = {}

for pid in holdout_df["id"].tolist():
    holdout_row = holdout_df[holdout_df["id"] == pid].iloc[0]
    patient_fixed = ou.load_patient_data(patient_id=pid, data_path=config.DATA_HOLDOUT)

    display(Markdown(f"---\n## Patient {pid}"))
    disp.display_patient_preop(patient_fixed, patient_id=pid, holdout_row=holdout_row)

    # Run GA for all scenarios
    all_results = {}
    for key in SCENARIOS:
        preset = runs.PRESETS[key]
        print(f"\n▶ Running: {preset['label']}...")

        result = runs.run_optimization(
            preset=preset,
            patient_fixed=patient_fixed,
            delta_bundles=delta_bundles,
            mech_fail_bundle=mech_fail_bundle,
            xl=xl,
            xu=xu,
            odi_bundle=ODI_bundle,
            pop_size=POP_SIZE,
            n_gen=N_GEN,
            seed=SEED,
            verbose=False,
            top_n=TOP_N,
            score_tolerance=SCORE_TOL,
            pso_ll_override=PSO_LL_OVERRIDE,
        )
        if result is None:
            continue
        all_results[key] = result
        print(f"  ✓ Best composite: {result['best_result']['composite_score']:.2f}"
              f"  |  Mech fail: {result['best_result']['mech_fail_prob']*100:.1f}%")

    print(f"\n{'='*60}")
    print(f"Completed {len(all_results)} scenarios.")

    # Build actual_x vector from holdout data
    actual_x = []
    for var_spec in config.DECISION_VAR_SPECS:
        col = var_spec["col_name"]
        val = holdout_row[col]
        if var_spec.get("categorical", False):
            actual_x.append(var_spec["choices"].index(val))
        else:
            actual_x.append(int(val))
    actual_x = np.array(actual_x)

    # Evaluate actual plan using real postop values
    actual_eval = ou.evaluate_actual_plan(
        actual_x, patient_fixed, holdout_row, mech_fail_bundle
    )

    # Best solution per scenario (Actual vs best optimized per scenario)
    disp.display_best_per_scenario(
        all_results, patient_fixed, actual_eval, SCENARIOS, deduplicate=DEDUP_BEST
    )

    # Store for reference
    all_patient_results[pid] = {
        "patient_fixed": patient_fixed,
        "all_results": all_results,
        "actual_eval": actual_eval,
        "actual_x": actual_x,
    }

    # --- 4 Best Solutions per Scenario (commented out as needed) ---
    # for key in SCENARIOS:
    #     if key not in all_results:
    #         continue
    #     r = all_results[key]
    #     preset = runs.PRESETS[key]
    #     display(Markdown(f"### {r['label']}"))
    #     display(Markdown(f"*{preset['description']}*"))
    #     disp.display_solutions_with_actual(r["diverse_df"], patient_fixed, actual_eval)


,sex,age,ASA_CLASS,revision,mech_fail_last,gap_score_preop,gap_category,gap_score_postop,gap_category_postop,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,ODI_preop
id,,,,,,,,,,,,,,,,,,,
1176294,MALE,71,NaN,1,1.0,10.0,SD,1.0,P,FS,lower,1,1,1,0,4,2,1,NaN
6309980,FEMALE,76,NaN,1,1.0,3.0,MD,4.0,MD,PS,higher,2,1,1,0,4,2,0,NaN
6516425,FEMALE,57,3.0,1,0.0,12.0,SD,7.0,SD,PS,higher,1,1,0,0,4,4,0,28.0
6606929,FEMALE,63,2.0,1,0.0,10.0,SD,5.0,MD,Hook,higher,3,0,1,0,4,2,1,6.0
7134904,FEMALE,72,2.0,1,0.0,13.0,SD,13.0,SD,Hook,higher,0,0,0,0,4,2,1,NaN
7141640,FEMALE,32,NaN,0,1.0,0.0,P,1.0,P,PS,higher,3,1,0,1,3,3,0,NaN
7287555,MALE,59,2.0,1,0.0,12.0,SD,6.0,MD,PS,lower,0,0,0,0,4,2,1,NaN
7361075,FEMALE,66,2.0,0,0.0,12.0,SD,6.0,MD,Hook,higher,2,1,0,0,3,3,0,NaN
7390166,MALE,58,2.0,0,0.0,4.0,MD,1.0,P,PS,lower,2,1,0,0,3,2,0,NaN


---
## Patient 1176294

**Patient 1176294 — Preop Profile**

Parameter,Value
DEMOGRAPHICS,
Age,71
Sex,MALE
BMI,28.0
ASA Class,–
CCI,–
Revision,Yes
Smoking,–
────────────,──────────
ALIGNMENT,



▶ Running: Equal Weights (Composite)...
  ✓ Best composite: 9.78  |  Mech fail: 51.9%

▶ Running: Minimize Mechanical Failure...
  ✓ Best composite: 37.13  |  Mech fail: 51.8%

▶ Running: Minimize Mechanical Failure + T4L1PA...
  ✓ Best composite: 25.39  |  Mech fail: 50.8%

▶ Running: Minimize L4S1 Penalty...
  ✓ Best composite: 10.16  |  Mech fail: 50.8%

▶ Running: Minimize T4L1PA Penalty...
  ✓ Best composite: 10.16  |  Mech fail: 50.8%

▶ Running: Minimize GAP Score...
  ✓ Best composite: 18.07  |  Mech fail: 51.9%

▶ Running: Minimize GAP Score and Mechanical Failure...
  ✓ Best composite: 32.07  |  Mech fail: 51.8%

▶ Running: Minimize LL (PI-LL) Penalty...
  ✓ Best composite: 10.94  |  Mech fail: 50.8%

▶ Running: Minimize Postop ODI...
  ⚠ Skipped: patient has no preop ODI

Completed 8 scenarios.


...

---
## Patient 7609811

**Patient 7609811 — Preop Profile**

Parameter,Value
DEMOGRAPHICS,
Age,67
Sex,FEMALE
BMI,19.4
ASA Class,–
CCI,–
Revision,No
Smoking,–
────────────,──────────
ALIGNMENT,



▶ Running: Equal Weights (Composite)...
  ✓ Best composite: 11.56  |  Mech fail: 10.9%

▶ Running: Minimize Mechanical Failure...
  ✓ Best composite: 11.01  |  Mech fail: 10.7%

▶ Running: Minimize Mechanical Failure + T4L1PA...
  ✓ Best composite: 5.33  |  Mech fail: 10.6%

▶ Running: Minimize L4S1 Penalty...
  ✓ Best composite: 2.13  |  Mech fail: 10.6%

▶ Running: Minimize T4L1PA Penalty...
  ✓ Best composite: 2.14  |  Mech fail: 10.7%

▶ Running: Minimize GAP Score...
  ✓ Best composite: 30.37  |  Mech fail: 10.7%

▶ Running: Minimize GAP Score and Mechanical Failure...
  ✓ Best composite: 23.73  |  Mech fail: 10.7%

▶ Running: Minimize LL (PI-LL) Penalty...
  ✓ Best composite: 2.89  |  Mech fail: 10.6%

▶ Running: Minimize Postop ODI...
  ⚠ Skipped: patient has no preop ODI

Completed 8 scenarios.


Parameter,Actual,Equal Weights (Composite),Minimize Mechanical Failure,Minimize Mechanical Failure + T4L1PA,Minimize L4S1 Penalty,Minimize T4L1PA Penalty,Minimize GAP Score,Minimize GAP Score and Mechanical Failure,Minimize LL (PI-LL) Penalty
Alignment Score,2.6,11.7,11.8,27.3,11.7,27.3,11.7,11.7,27.7
Optimization Score,2.6,11.6,11.1,5.3,2.1,2.1,30.4,23.7,2.9
Mech Fail Prob,17.8%,10.9%,10.7%,10.6%,10.7%,10.7%,10.9%,10.7%,10.7%
Predicted ODI,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
GAP Score,13.0 (SD) → 2 (P),13.0 (SD) → 5 (MD),13.0 (SD) → 5 (MD),13.0 (SD) → 8 (SD),13.0 (SD) → 5 (MD),13.0 (SD) → 8 (SD),13.0 (SD) → 5 (MD),13.0 (SD) → 5 (MD),13.0 (SD) → 8 (SD)
────────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────
SURGICAL PLAN,,,,,,,,,
UIV_implant,Hook,FS,FS,FS,FS,FS,FS,FS,FS
num_levels_cat,higher,higher,lower,higher,higher,higher,higher,higher,higher
num_interbody_fusion_levels,2,5,5,5,5,5,5,5,5
